In [1]:
import duckdb

con = duckdb.connect('database.duckdb')

In [2]:
# Consultas de analise de gastos publicos
queries = {
    "overview_por_ano": """
SELECT ano, COUNT(*) AS qtd_empenhos,
       SUM(valorEmpenhado) AS total_empenhado,
       SUM(valorLiquidado) AS total_liquidado,
       SUM(valorPago) AS total_pago
FROM empenhos
GROUP BY ano
ORDER BY ano DESC
""",
    "overview_por_orgao": """
SELECT nomeOrgao, COUNT(*) AS qtd_empenhos,
       SUM(valorEmpenhado) AS total_empenhado,
       SUM(valorPago) AS total_pago
FROM empenhos
GROUP BY nomeOrgao
ORDER BY total_pago DESC
LIMIT 20
""",
    "evolucao_mensal": """
SELECT ano, mes, COUNT(*) AS qtd_empenhos,
       SUM(valorEmpenhado) AS total_empenhado,
       SUM(valorPago) AS total_pago
FROM empenhos
GROUP BY ano, mes
ORDER BY ano DESC, mes DESC
""",
    "sazonalidade_mensal": """
SELECT mes, COUNT(*) AS qtd_empenhos,
       SUM(valorEmpenhado) AS total_empenhado,
       ROUND(AVG(valorEmpenhado), 2) AS media_empenho
FROM empenhos
GROUP BY mes
ORDER BY total_empenhado DESC
""",
    "execucao_orcamentaria": """
SELECT ano,
       SUM(valorEmpenhado) AS total_empenhado,
       SUM(valorLiquidado) AS total_liquidado,
       SUM(valorPago) AS total_pago,
       ROUND(100.0 * SUM(valorLiquidado) / NULLIF(SUM(valorEmpenhado), 0), 2) AS pct_liquidado,
       ROUND(100.0 * SUM(valorPago) / NULLIF(SUM(valorEmpenhado), 0), 2) AS pct_pago
FROM empenhos
GROUP BY ano
ORDER BY ano DESC
""",
    "contratos_com_aditivos": """
SELECT numeroContrato, nomeOrgao, objeto, valorOriginal,
       valorAditivos, valorApostilas, valorTotal,
       ROUND(100.0 * (valorAditivos + valorApostilas) / NULLIF(valorOriginal, 0), 2) AS pct_aditivo
FROM contratos
WHERE valorAditivos > 0 OR valorApostilas > 0
ORDER BY (valorAditivos + valorApostilas) DESC
LIMIT 20
""",
    "maiores_fornecedores": """
SELECT contratado, cnpjCpf, COUNT(*) AS qtd_contratos,
       SUM(valorTotal) AS total_contratado
FROM contratos
GROUP BY contratado, cnpjCpf
ORDER BY total_contratado DESC
LIMIT 20
""",
    "diferenca_estimado_adjudicado": """
SELECT numeroProcesso, nomeOrgao, objeto, valorEstimado,
       valorAdjudicado, valorAdjudicado - valorEstimado AS diferenca,
       ROUND(100.0 * (valorAdjudicado - valorEstimado) / NULLIF(valorEstimado, 0), 2) AS pct_diferenca
FROM contratacoes
WHERE valorEstimado > 0
ORDER BY diferenca DESC
LIMIT 20
""",
    "licitacoes_baixa_competicao": """
SELECT numeroProcesso, nomeOrgao, objeto, modalidade,
       numParticipantes, valorAdjudicado
FROM contratacoes
WHERE numParticipantes <= 2
  AND modalidade NOT IN ('Dispensa', 'Convite')
ORDER BY valorAdjudicado DESC
LIMIT 20
""",
    "processos_sem_contrato": """
SELECT COUNT(DISTINCT c.numeroProcesso) AS total_processos_contratacao,
       COUNT(DISTINCT ct.numeroProcessoLicitacao) AS total_processos_com_contrato,
       COUNT(DISTINCT c.numeroProcesso) - COUNT(DISTINCT ct.numeroProcessoLicitacao) AS processos_sem_contrato
FROM contratacoes c
LEFT JOIN contratos ct ON c.numeroProcesso = ct.numeroProcessoLicitacao
""",
    "empenhos_sem_contrato": """
SELECT COUNT(*) AS qtd_empenhos_sem_contrato,
       SUM(valorEmpenhado) AS valor_sem_contrato
FROM empenhos
WHERE contrato IS NULL OR contrato = ''
""",
    "performance_por_modalidade": """
SELECT modalidade, COUNT(*) AS qtd_licitacoes,
       SUM(valorEstimado) AS valor_estimado_total,
       SUM(valorAdjudicado) AS valor_adjudicado_total,
       ROUND(AVG(valorAdjudicado), 2) AS valor_medio_adjudicado,
       ROUND(100.0 * SUM(valorAdjudicado) / SUM(NULLIF(valorEstimado, 0)), 2) AS pct_estimado
FROM contratacoes
GROUP BY modalidade
ORDER BY qtd_licitacoes DESC
""",
    "tempo_medio_por_modalidade": """
SELECT modalidade, COUNT(*) AS qtd_processos,
       ROUND(AVG(CAST(dataAdjudicacao - dataAbertura AS INTEGER)), 1) AS dias_medio
FROM contratacoes
WHERE dataAbertura IS NOT NULL AND dataAdjudicacao IS NOT NULL
GROUP BY modalidade
ORDER BY dias_medio DESC
""",
    "folha_por_orgao_periodo": """
SELECT anoExercicio, mesReferencia, orgaoLotacao,
       COUNT(DISTINCT cpfServidor) AS qtd_servidores,
       ROUND(SUM(valorBruto), 2) AS total_bruto,
       ROUND(SUM(valorLiquido), 2) AS total_liquido,
       ROUND(AVG(valorBruto), 2) AS media_bruto
FROM servidores
GROUP BY anoExercicio, mesReferencia, orgaoLotacao
ORDER BY anoExercicio DESC, mesReferencia DESC
LIMIT 20
""",
    "distribuicao_salarial_cargo": """
SELECT nomeCargo, tipoCargo, COUNT(*) AS qtd_servidores,
       ROUND(AVG(valorBruto), 2) AS salario_medio,
       ROUND(MIN(valorBruto), 2) AS salario_min,
       ROUND(MAX(valorBruto), 2) AS salario_max
FROM servidores
GROUP BY nomeCargo, tipoCargo
ORDER BY salario_medio DESC
LIMIT 20
""",
    "liquidacao_por_natureza": """
SELECT codigoNatureza, COUNT(*) AS qtd_liquidacoes,
       SUM(valorEmpenhado) AS valor_empenhado,
       SUM(valorGD) AS valor_gd,
       SUM(valorAnulado) AS valor_anulado,
       ROUND(100.0 * SUM(valorGD) / SUM(NULLIF(valorEmpenhado, 0)), 2) AS taxa_liquidacao
FROM liquidacoes
GROUP BY codigoNatureza
ORDER BY valor_empenhado DESC
LIMIT 20
""",
    "diferenca_empenho_liquidacao": """
SELECT l.nomeOrgao, l.numeroEmpenho, l.valorEmpenhado,
       l.valorGD AS valor_liquidado,
       l.valorEmpenhado - l.valorGD AS diferenca,
       COUNT(*) AS qtd_registros_liquidacao
FROM liquidacoes l
GROUP BY l.nomeOrgao, l.numeroEmpenho, l.valorEmpenhado, l.valorGD
HAVING l.valorEmpenhado - l.valorGD > 0
ORDER BY diferenca DESC
LIMIT 20
""",
    "fluxo_licitacao_pagamento": """
SELECT EXTRACT(YEAR FROM c.dataCriacao) AS ano,
       COUNT(DISTINCT c.numeroProcesso) AS licitacoes,
       COUNT(DISTINCT ct.numeroContrato) AS contratos,
       COUNT(DISTINCT e.numeroEmpenho) AS empenhos,
       COUNT(DISTINCT CASE WHEN e.valorPago > 0 THEN e.numeroEmpenho END) AS empenhos_pagos,
       SUM(c.valorAdjudicado) AS valor_adjudicado_total,
       SUM(e.valorPago) AS valor_pago_total
FROM contratacoes c
LEFT JOIN contratos ct ON c.numeroProcesso = ct.numeroProcessoLicitacao
LEFT JOIN empenhos e ON ct.numeroContrato = e.contrato
GROUP BY EXTRACT(YEAR FROM c.dataCriacao)
ORDER BY ano DESC
""",
    "empenhos_outliers": """
WITH stats AS (
    SELECT AVG(valorEmpenhado) AS media,
           STDDEV(valorEmpenhado) AS desvio
    FROM empenhos
)
SELECT e.ano, e.nomeOrgao, e.numeroEmpenho,
       e.descricaoEmpenho, e.valorEmpenhado,
       s.media,
       ROUND((e.valorEmpenhado - s.media) / NULLIF(s.desvio, 0), 2) AS desvios_padrao
FROM empenhos e, stats s
WHERE e.valorEmpenhado > s.media + 3 * s.desvio
ORDER BY e.valorEmpenhado DESC
LIMIT 20
""",
    "fornecedores_multiplos_orgaos": """
SELECT contratado, cnpjCpf,
       COUNT(DISTINCT nomeOrgao) AS qtd_orgaos_diferentes,
       COUNT(*) AS qtd_contratos,
       SUM(valorTotal) AS total_contratado
FROM contratos
GROUP BY contratado, cnpjCpf
HAVING COUNT(DISTINCT nomeOrgao) > 5
ORDER BY qtd_contratos DESC
""",
    "processos_pendentes_antigos": """
SELECT numeroProcesso, nomeOrgao, objeto, modalidade,
       situacao, dataCriacao,
       DATEDIFF('day', dataCriacao, CURRENT_DATE) AS dias_pendente
FROM contratacoes
WHERE situacao NOT IN ('Homologada', 'Encerrada', 'Finalizada')
ORDER BY dias_pendente DESC
LIMIT 20
"""
}


def executar_consulta(nome):
    """Executa uma consulta cadastrada e retorna um DataFrame."""
    if nome not in queries:
        raise KeyError(f"Consulta inexistente: {nome}")
    return con.sql(queries[nome]).df()


# Exemplo: executar_consulta("overview_por_ano")

In [3]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd

CORES = {
    "empenhado": "#1f77b4",
    "liquidado": "#ff7f0e",
    "pago":      "#2ca02c",
    "alerta":    "#d62728",
    "neutro":    "#7f7f7f",
}

In [4]:
def plot_overview_por_ano(df):
    """Barras agrupadas + linha de qtd empenhos."""
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    for col, cor in [("total_empenhado", CORES["empenhado"]),
                     ("total_liquidado", CORES["liquidado"]),
                     ("total_pago", CORES["pago"])]:
        fig.add_bar(x=df["ano"], y=df[col], name=col, marker_color=cor)
    fig.add_trace(
        go.Scatter(x=df["ano"], y=df["qtd_empenhos"],
                   name="Qtd empenhos", mode="lines+markers",
                   line=dict(color="black", dash="dot")),
        secondary_y=True,
    )
    fig.update_layout(barmode="group", title="Execução orçamentária por ano",
                      yaxis_title="Valor (R$)", yaxis2_title="Qtd empenhos")
    return fig


def plot_overview_por_orgao(df):
    """Barras horizontais ordenadas."""
    fig = px.bar(df.sort_values("total_pago"),
                 x="total_pago", y="nomeOrgao", orientation="h",
                 title="Top 20 órgãos por valor pago",
                 labels={"total_pago": "Valor pago (R$)", "nomeOrgao": ""},
                 color="total_pago", color_continuous_scale="Blues")
    fig.update_layout(yaxis=dict(tickfont=dict(size=10)))
    return fig

def plot_evolucao_mensal(df):
    """Linha temporal (ano-mês)."""
    df = df.copy()
    df["periodo"] = df["ano"].astype(str) + "-" + df["mes"].astype(str).str.zfill(2)
    df = df.sort_values("periodo")
    fig = px.line(df, x="periodo", y=["total_empenhado", "total_pago"],
                  markers=True, title="Evolução mensal de empenhos e pagamentos")
    fig.update_layout(xaxis_title="", yaxis_title="Valor (R$)")
    return fig


def plot_sazonalidade_mensal(df):
    """Barras + linha (média)."""
    df = df.sort_values("mes")
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_bar(x=df["mes"], y=df["total_empenhado"],
                name="Total empenhado", marker_color=CORES["empenhado"])
    fig.add_trace(
        go.Scatter(x=df["mes"], y=df["media_empenho"], name="Média empenho",
                   mode="lines+markers", line=dict(color=CORES["alerta"])),
        secondary_y=True,
    )
    fig.update_layout(title="Sazonalidade mensal dos empenhos",
                      xaxis_title="Mês", yaxis_title="Total (R$)",
                      yaxis2_title="Média (R$)")
    return fig
def plot_execucao_orcamentaria(df):
    """Barras empilhadas + percentuais anotados."""
    df = df.sort_values("ano")
    fig = go.Figure()
    fig.add_bar(x=df["ano"], y=df["total_pago"],     name="Pago",      marker_color=CORES["pago"])
    fig.add_bar(x=df["ano"], y=df["total_liquidado"] - df["total_pago"],
                name="Liquidado não pago", marker_color=CORES["liquidado"])
    fig.add_bar(x=df["ano"], y=df["total_empenhado"] - df["total_liquidado"],
                name="Empenhado não liquidado", marker_color=CORES["empenhado"])
    for _, r in df.iterrows():
        fig.add_annotation(x=r["ano"], y=r["total_empenhado"],
                           text=f"{r['pct_pago']:.1f}% pago",
                           showarrow=False, yshift=10, font=dict(size=10))
    fig.update_layout(barmode="stack", title="Composição da execução por ano",
                      yaxis_title="Valor (R$)")
    return fig
def plot_contratos_com_aditivos(df):
    """Barras horizontais com % aditivo no hover."""
    df = df.copy()
    df["label"] = df["numeroContrato"].astype(str) + " — " + df["nomeOrgao"].str[:25]
    fig = px.bar(df.sort_values("pct_aditivo"),
                 x="pct_aditivo", y="label", orientation="h",
                 color="pct_aditivo", color_continuous_scale="Reds",
                 hover_data=["valorOriginal", "valorAditivos",
                             "valorApostilas", "valorTotal"],
                 title="Contratos com maior % de aditivos")
    fig.update_layout(xaxis_title="% aditivo sobre valor original", yaxis_title="")
    return fig


def plot_maiores_fornecedores(df):
    """Treemap por fornecedor."""
    fig = px.treemap(df, path=["contratado"], values="total_contratado",
                     color="qtd_contratos", color_continuous_scale="Viridis",
                     title="Top 20 maiores fornecedores")
    return fig


def plot_fornecedores_multiplos_orgaos(df):
    """Scatter: nº de órgãos × nº de contratos, tamanho = valor."""
    fig = px.scatter(df, x="qtd_orgaos_diferentes", y="qtd_contratos",
                     size="total_contratado", color="total_contratado",
                     hover_name="contratado", color_continuous_scale="Turbo",
                     title="Fornecedores com presença em múltiplos órgãos")
    return fig
def plot_diferenca_estimado_adjudicado(df):
    """Waterfall simplificado em barras divergentes."""
    df = df.sort_values("pct_diferenca")
    cores = [CORES["alerta"] if v > 0 else CORES["pago"] for v in df["pct_diferenca"]]
    fig = go.Figure(go.Bar(
        x=df["pct_diferenca"],
        y=df["numeroProcesso"],
        orientation="h",
        marker_color=cores,
        customdata=df[["valorEstimado", "valorAdjudicado", "nomeOrgao"]],
        hovertemplate="%{y}<br>%{customdata[2]}<br>"
                      "Estimado: R$ %{customdata[0]:,.2f}<br>"
                      "Adjudicado: R$ %{customdata[1]:,.2f}<br>"
                      "%{x:.2f}%<extra></extra>",
    ))
    fig.update_layout(title="Processos com maior desvio (adjudicado vs estimado)",
                      xaxis_title="% diferença", yaxis_title="")
    return fig


def plot_licitacoes_baixa_competicao(df):
    """Barra horizontal colorida por modalidade."""
    fig = px.bar(df.sort_values("valorAdjudicado"),
                 x="valorAdjudicado", y="numeroProcesso",
                 color="modalidade", orientation="h",
                 hover_data=["nomeOrgao", "numParticipantes"],
                 title="Licitações com baixa competição (≤2 participantes)")
    fig.update_layout(xaxis_title="Valor adjudicado (R$)", yaxis_title="")
    return fig


def plot_performance_por_modalidade(df):
    """Barras agrupadas estimado vs adjudicado + scatter de qtd."""
    df = df.sort_values("qtd_licitacoes", ascending=False)
    fig = go.Figure()
    fig.add_bar(x=df["modalidade"], y=df["valor_estimado_total"],
                name="Estimado", marker_color=CORES["empenhado"])
    fig.add_bar(x=df["modalidade"], y=df["valor_adjudicado_total"],
                name="Adjudicado", marker_color=CORES["pago"])
    fig.update_layout(barmode="group",
                      title="Performance por modalidade de licitação",
                      yaxis_title="Valor (R$)")
    return fig


def plot_tempo_medio_por_modalidade(df):
    """Barras horizontais ordenadas por tempo."""
    fig = px.bar(df.sort_values("dias_medio"),
                 x="dias_medio", y="modalidade", orientation="h",
                 color="dias_medio", color_continuous_scale="OrRd",
                 text="dias_medio",
                 title="Tempo médio (dias) entre abertura e adjudicação")
    fig.update_traces(texttemplate="%{text:.1f}", textposition="outside")
    fig.update_layout(xaxis_title="Dias", yaxis_title="")
    return fig

def plot_folha_por_orgao_periodo(df):
    """Heatmap órgão × período com valor líquido."""
    df = df.copy()
    df["periodo"] = df["anoExercicio"].astype(str) + "-" + \
                    df["mesReferencia"].astype(str).str.zfill(2)
    pivot = df.pivot_table(index="orgaoLotacao", columns="periodo",
                           values="total_liquido", aggfunc="sum")
    fig = px.imshow(pivot, aspect="auto", color_continuous_scale="YlGnBu",
                    title="Folha líquida por órgão e período")
    fig.update_layout(xaxis_title="", yaxis_title="")
    return fig


def plot_distribuicao_salarial_cargo(df):
    """Barras de intervalo (min-média-max) por cargo."""
    df = df.sort_values("salario_medio").tail(20)
    fig = go.Figure()
    for _, r in df.iterrows():
        fig.add_trace(go.Scatter(
            x=[r["salario_min"], r["salario_max"]],
            y=[r["nomeCargo"]] * 2,
            mode="lines", line=dict(color=CORES["neutro"], width=2),
            showlegend=False, hoverinfo="skip",
        ))
    fig.add_trace(go.Scatter(
        x=df["salario_medio"], y=df["nomeCargo"],
        mode="markers", marker=dict(size=12, color=CORES["empenhado"]),
        name="Média",
        error_x=dict(
            type="data",
            symmetric=False,
            array=df["salario_max"] - df["salario_medio"],
            arrayminus=df["salario_medio"] - df["salario_min"],
        ),
    ))
    fig.update_layout(title="Distribuição salarial por cargo (min/média/max)",
                      xaxis_title="Valor bruto (R$)", yaxis_title="")
    return fig

def plot_liquidacao_por_natureza(df):
    """Barras horizontais empilhadas + linha de taxa."""
    df = df.sort_values("valor_empenhado").tail(20)
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_bar(y=df["codigoNatureza"], x=df["valor_gd"],
                name="Liquidado (GD)", orientation="h",
                marker_color=CORES["pago"])
    fig.add_bar(y=df["codigoNatureza"],
                x=df["valor_empenhado"] - df["valor_gd"],
                name="Não liquidado", orientation="h",
                marker_color=CORES["neutro"])
    fig.add_trace(
        go.Scatter(y=df["codigoNatureza"], x=df["taxa_liquidacao"],
                   mode="markers", name="Taxa (%)",
                   marker=dict(color=CORES["alerta"], size=10)),
        secondary_y=True,
    )
    fig.update_layout(barmode="stack",
                      title="Liquidação por natureza de despesa",
                      xaxis_title="Valor (R$)")
    fig.update_yaxes(title="", secondary_y=False)
    fig.update_yaxes(title="Taxa (%)", secondary_y=True, range=[0, 105])
    return fig


def plot_diferenca_empenho_liquidacao(df):
    """Top diferenças empenho − liquidado."""
    df = df.sort_values("diferenca", ascending=True).tail(20)
    df["label"] = df["numeroEmpenho"].astype(str) + " — " + df["nomeOrgao"].str[:20]
    fig = px.bar(df, x="diferenca", y="label", orientation="h",
                 color="diferenca", color_continuous_scale="Reds",
                 title="Empenhos com maior saldo não liquidado")
    fig.update_layout(xaxis_title="Empenhado − Liquidado (R$)", yaxis_title="")
    return fig

def plot_fluxo_licitacao_pagamento(df):
    """Sankey simplificado em barras agrupadas por ano."""
    df = df.sort_values("ano")
    fig = go.Figure()
    for col, cor in [("licitacoes", "#1f77b4"),
                     ("contratos", "#ff7f0e"),
                     ("empenhos", "#2ca02c"),
                     ("empenhos_pagos", "#9467bd")]:
        fig.add_bar(x=df["ano"], y=df[col], name=col, marker_color=cor)
    fig.update_layout(barmode="group",
                      title="Fluxo licitação → contrato → empenho → pagamento",
                      yaxis_title="Quantidade")
    return fig


def plot_empenhos_outliers(df):
    """Scatter z-score por órgão, anotando descrição."""
    fig = px.scatter(df, x="valorEmpenhado", y="desvios_padrao",
                     color="nomeOrgao", size="valorEmpenhado",
                     hover_data=["numeroEmpenho", "descricaoEmpenho"],
                     title="Empenhos outliers (>3 desvios padrão)")
    fig.add_hline(y=3, line_dash="dash", line_color=CORES["alerta"])
    fig.update_layout(xaxis_title="Valor empenhado (R$)",
                      yaxis_title="Desvios padrão da média",
                      xaxis_type="log")
    return fig


def plot_processos_pendentes_antigos(df):
    """Barras horizontais coloridas por dias pendentes."""
    df = df.sort_values("dias_pendente")
    fig = px.bar(df, x="dias_pendente", y="numeroProcesso",
                 color="dias_pendente", color_continuous_scale="Reds",
                 orientation="h",
                 hover_data=["nomeOrgao", "modalidade", "situacao", "dataCriacao"],
                 title="Processos pendentes mais antigos")
    fig.update_layout(xaxis_title="Dias pendentes", yaxis_title="")
    return fig

def plot_processos_sem_contrato(df):
    """Indicador único em gauge."""
    row = df.iloc[0]
    total = row["total_processos_contratacao"]
    sem = row["processos_sem_contrato"]
    pct = 100 * sem / total if total else 0
    fig = go.Figure(go.Indicator(
        mode="gauge+number+delta",
        value=pct,
        number={"suffix": "%"},
        title={"text": f"Processos sem contrato<br>"
                       f"<sub>{sem:,} de {total:,}</sub>"},
        gauge={"axis": {"range": [0, 100]},
               "bar": {"color": CORES["alerta"]},
               "steps": [
                   {"range": [0, 10], "color": "#d4edda"},
                   {"range": [10, 25], "color": "#fff3cd"},
                   {"range": [25, 100], "color": "#f8d7da"},
               ]},
    ))
    fig.update_layout(height=350)
    return fig


def plot_empenhos_sem_contrato(df):
    """Indicador simples."""
    row = df.iloc[0]
    fig = go.Figure(go.Indicator(
        mode="number",
        value=row["valor_sem_contrato"],
        number={"prefix": "R$ ", "valueformat": ",.2f"},
        title={"text": f"Empenhos sem contrato<br>"
                       f"<sub>{row['qtd_empenhos_sem_contrato']:,} registros</sub>"},
    ))
    fig.update_layout(height=300)
    return fig

PLOTS = {
    "overview_por_ano":              plot_overview_por_ano,
    "overview_por_orgao":            plot_overview_por_orgao,
    "evolucao_mensal":               plot_evolucao_mensal,
    "sazonalidade_mensal":           plot_sazonalidade_mensal,
    "execucao_orcamentaria":         plot_execucao_orcamentaria,
    "contratos_com_aditivos":        plot_contratos_com_aditivos,
    "maiores_fornecedores":          plot_maiores_fornecedores,
    "diferenca_estimado_adjudicado": plot_diferenca_estimado_adjudicado,
    "licitacoes_baixa_competicao":   plot_licitacoes_baixa_competicao,
    "processos_sem_contrato":        plot_processos_sem_contrato,
    "empenhos_sem_contrato":         plot_empenhos_sem_contrato,
    "performance_por_modalidade":    plot_performance_por_modalidade,
    "tempo_medio_por_modalidade":    plot_tempo_medio_por_modalidade,
    "folha_por_orgao_periodo":       plot_folha_por_orgao_periodo,
    "distribuicao_salarial_cargo":   plot_distribuicao_salarial_cargo,
    "liquidacao_por_natureza":       plot_liquidacao_por_natureza,
    "diferenca_empenho_liquidacao":  plot_diferenca_empenho_liquidacao,
    "fluxo_licitacao_pagamento":     plot_fluxo_licitacao_pagamento,
    "empenhos_outliers":             plot_empenhos_outliers,
    "fornecedores_multiplos_orgaos": plot_fornecedores_multiplos_orgaos,
    "processos_pendentes_antigos":   plot_processos_pendentes_antigos,
}


def gerar_plot(nome: str, df=None):
    """Gera o plot correspondente a uma query.
    
    Uso:
        fig = gerar_plot("overview_por_ano")
        fig.show()
    """
    if nome not in PLOTS:
        raise KeyError(f"Sem plot definado para: {nome}")
    if df is None:
        df = executar_consulta(nome)   # do seu arquivo original
    return PLOTS[nome](df)


def gerar_dashboard_html(saida="dashboard.html"):
    """Gera um HTML único com todos os plots."""
    from plotly.io import write_html
    figs = []
    for nome in PLOTS:
        try:
            figs.append(gerar_plot(nome))
        except Exception as e:
            print(f"[skip] {nome}: {e}")
    with open(saida, "w", encoding="utf-8") as f:
        f.write("<html><head><meta charset='utf-8'>"
                "<title>Dashboard Gastos Públicos</title></head><body>"
                "<h1>Análise de Gastos Públicos</h1>")
        for fig, nome in zip(figs, PLOTS):
            f.write(f"<h2>{nome}</h2>")
            f.write(fig.to_html(full_html=False, include_plotlyjs="cdn"))
        f.write("</body></html>")
    print(f"Dashboard salvo em {saida}")

In [5]:
gerar_dashboard_html()

Dashboard salvo em dashboard.html
